# SatQuery AI — Round 1: Remote Sensing Domain Warm-up (RSVQA-LR)
### Smart India Hackathon (SIH 26167) | ISRO Space Technology Theme

**Objective**: Teach the VLM basic satellite vocabulary and image patterns before the harder tasks.
- **Dataset**: RSVQA-LR (772 images, simple binary land-cover questions)
- **Architecture**: InternVL3-1B (4-bit QLoRA) + S1/S2 Encoders + Projection Heads
- **Duration**: ~10–12 minutes on Free Google Colab T4 GPU
- **Output**: `MyDrive/SatQuery_AI/ckpt/r1_warmup/best` adapter

In [12]:
# 1. GPU Check
!nvidia-smi
import torch
assert torch.cuda.is_available(), "Please enable GPU in Runtime > Change runtime type > T4 GPU"

Sat Sep  5 18:48:25 2026       
+-----------------------------------------------------------------------------------------+
| NVIDIA-SMI 580.82.07              Driver Version: 580.82.07      CUDA Version: 13.0     |
+-----------------------------------------+------------------------+----------------------+
| GPU  Name                 Persistence-M | Bus-Id          Disp.A | Volatile Uncorr. ECC |
| Fan  Temp   Perf          Pwr:Usage/Cap |           Memory-Usage | GPU-Util  Compute M. |
|                                         |                        |               MIG M. |
|=========================================+========================+======================|
|   0  Tesla T4                       Off |   00000000:00:04.0 Off |                    0 |
| N/A   40C    P8             13W /   70W |       3MiB /  15360MiB |      0%      Default |
|                                         |                        |                  N/A |
+-----------------------------------------+-----

In [13]:
# 2. Mount Google Drive
from google.colab import drive
drive.mount('/content/drive')
import os
assert os.path.exists('/content/drive/MyDrive/SatQuery_AI'), "Run 00_drive_setup.ipynb and 01_download_datasets.ipynb first!"

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


In [14]:
# 3. Clone / Update Repository
import os
if not os.path.exists("/content/SIH"):
    !git clone https://github.com/abhineet115/SIH.git /content/SIH
else:
    !cd /content/SIH && git pull
%cd /content/SIH
!pip install -q -r training/requirements_colab.txt

Already up to date.
/content/SIH


In [26]:
!cd /content/SIH && git pull origin main

remote: Enumerating objects: 11, done.
remote: Counting objects: 100% (11/11), done.
remote: Compressing objects: 100% (1/1), done.
remote: Total 6 (delta 5), reused 6 (delta 5), pack-reused 0 (from 0)
Unpacking objects: 100% (6/6), 1.44 KiB | 491.00 KiB/s, done.
From https://github.com/abhineet115/SIH
 * branch            main       -> FETCH_HEAD
   7358362..45e0785  main       -> origin/main
Updating 7358362..45e0785
Fast-forward
 training/export.py             | 25 ++++++++----
 training/models/rs_internvl.py | 88 +++++++++++++++++++++++++++---------------
 2 files changed, 75 insertions(+), 38 deletions(-)


In [27]:
# 4. Launch Round 1 Training (Auto-resumes if interrupted)
!python training/train_round.py \
    --round 1 \
    --drive /content/drive/MyDrive/SatQuery_AI \
    --base-model OpenGVLab/InternVL3-1B


  ROUND 1: R1_WARMUP
  Task: binary_vqa | Dataset: rsvqa_lr
  Output: /content/drive/MyDrive/SatQuery_AI/ckpt/r1_warmup

[Round 1] Starting fresh (no previous adapter)
[RSInternVL] Loading InternVL3-1B...
[transformers] `torch_dtype` is deprecated! Use `dtype` instead!
FlashAttention2 is not installed.
Loading weights: 100% 637/637 [00:01<00:00, 568.82it/s]
[RSInternVL] Detected LLM embedding dimension: 896
[RSInternVL] Loading S1 ViT encoder (frozen)...
[S1ViTEncoder] Fallback: generic ViT-Base
[S1ViTEncoder] Frozen ✓
[RSInternVL] Loading S2 ViT encoder (frozen)...
[S2ViTEncoder] Fallback: generic ViT-Base
[S2ViTEncoder] Frozen ✓
trainable params: 1,081,344 || all params: 630,779,264 || trainable%: 0.1714
[RSInternVL] Ready. Task mode: binary_vqa
[RSVQA] Loaded 772 samples from /content/drive/MyDrive/SatQuery_AI/datasets/rsvqa_lr_train.json
[RSVQA] Loaded 150 samples from /content/drive/MyDrive/SatQuery_AI/datasets/rsvqa_lr_val.json
2026-09-05 19:02:39.602888: I tensorflow/core/platf

In [29]:
# 5. Verify Checkpoint in Google Drive
import os
best_path = "/content/drive/MyDrive/SatQuery_AI/ckpt/r1_warmup/best"
if os.path.exists(best_path):
    print(f"✅ Round 1 Success! Adapter saved at: {best_path}")
    print(os.listdir(best_path))
else:
    print("❌ Checkpoint not found. Please review training logs above.")

✅ Round 1 Success! Adapter saved at: /content/drive/MyDrive/SatQuery_AI/ckpt/r1_warmup/best
['README.md', 'adapter_model.safetensors', 'adapter_config.json', 'projection_heads.pt', 'chat_template.jinja', 'tokenizer_config.json', 'tokenizer.json']
